In [8]:
import numpy as np
import pandas as pd

### 传入loc的切片在python中是特殊情况，**包含右边**
eg：df.loc[1:3, "Party" : "%"]  # 包含1、2、3行和%列

### 传入iloc的切片在python中是常规情况，**不包含右边**

In [9]:
elections = pd.read_csv("data/elections.csv")
elections.head()

,Year,Candidate,Party,Popular vote,Result,%
0,1824,Andrew Jackson,Democratic-Republican,151271,loss,57.210122
1,1824,John Quincy Adams,Democratic-Republican,113142,win,42.789878
2,1828,Andrew Jackson,Democratic,642806,win,56.203927
3,1828,John Quincy Adams,National Republican,500897,loss,43.796073
4,1832,Andrew Jackson,Democratic,702735,win,54.574789


In [10]:
elections.iloc[1:3, 1]

1    John Quincy Adams
2       Andrew Jackson
Name: Candidate, dtype: object

In [11]:
elections.loc[1:3, "Party": "%"]

,Party,Popular vote,Result,%
1,Democratic-Republican,113142,win,42.789878
2,Democratic,642806,win,56.203927
3,National Republican,500897,loss,43.796073


### 上下文依赖提取[]，结合loc和iloc，只接受一个参数：行号的切片、列标签的列表、单独的列标签
要么对行进行过滤，要么对列进行过滤<br>
传入的是整数或整数切片时，认为是行标签<br>
传入的是字符串时，认为是列标签<br>
这里的切片是不包含右边的


In [12]:
elections[['Year' , 'Party']]  # 列标签切片

,Year,Party
0,1824,Democratic-Republican
1,1824,Democratic-Republican
2,1828,Democratic
3,1828,National Republican
4,1832,Democratic
...,...,...
177,2016,Green
178,2020,Democratic
179,2020,Republican
180,2020,Libertarian


In [13]:
import urllib.request
import os.path
import zipfile

data_url = "https://www.ssa.gov/oact/babynames/state/namesbystate.zip"
local_filename = "data/babynamesbystate.zip"
if not os.path.exists(local_filename): # If the data exists don't download again
    with urllib.request.urlopen(data_url) as resp, open(local_filename, 'wb') as f:
        f.write(resp.read())

zf = zipfile.ZipFile(local_filename, 'r')

ca_name = 'STATE.CA.TXT'
field_names = ['State', 'Sex', 'Year', 'Name', 'Count']
with zf.open(ca_name) as fh:
    babynames = pd.read_csv(fh, header=None, names=field_names)

babynames.head()

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134


## 条件提取
可以把条件的布尔序列传递给**.loc**和**上下文依赖提取的[]操作符**


In [14]:
babynames_first_10_rows = babynames.loc[:9, :]

babynames_first_10_rows

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134
5,CA,F,1910,Ruth,128
6,CA,F,1910,Evelyn,126
7,CA,F,1910,Alice,118
8,CA,F,1910,Virginia,101
9,CA,F,1910,Elizabeth,93


上下文依赖提取的[]操作符示例<br>
提供手动创建条件提取的布尔序列，传入后，返回babynames_first_10_rows只保留布尔值为真的部分<br>
**返回的dataframe中行的索引不会改变，如果需要改变要通过reset_index重置索引**<br>
这里原本的babynames_first_10_rows并没有改变

In [15]:
babynames_first_10_rows[[True, False, True, False, True, False, True, False, True, False]]

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
2,CA,F,1910,Dorothy,220
4,CA,F,1910,Frances,134
6,CA,F,1910,Evelyn,126
8,CA,F,1910,Virginia,101


.loc的示例<br>


In [17]:
babynames_first_10_rows.loc[[True, False, True, False, True, False, True, False, True, False]]

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
2,CA,F,1910,Dorothy,220
4,CA,F,1910,Frances,134
6,CA,F,1910,Evelyn,126
8,CA,F,1910,Virginia,101


创建逻辑运算符序列对象作为布尔序列

In [22]:
logical_operator = (babynames["Sex"] == "F")
babynames.loc[logical_operator, :]
# babynames.loc[logical_operator] 效果相同，因为.loc中第二操作数不是必要的，如果省略，默认为需要所有列

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134
...,...,...,...,...,...
239532,CA,F,2022,Zemira,5
239533,CA,F,2022,Ziggy,5
239534,CA,F,2022,Zimal,5
239535,CA,F,2022,Zosia,5


#### 通过位运算符得到复合条件
<img src="./images/位运算符示例.png">

In [25]:
babynames[(babynames["Sex"] == "F") | (babynames["Year"] > 2000)]

,State,Sex,Year,Name,Count
0,CA,F,1910,Mary,295
1,CA,F,1910,Helen,239
2,CA,F,1910,Dorothy,220
3,CA,F,1910,Margaret,163
4,CA,F,1910,Frances,134
...,...,...,...,...,...
407423,CA,M,2022,Zayvier,5
407424,CA,M,2022,Zia,5
407425,CA,M,2022,Zora,5
407426,CA,M,2022,Zuriel,5


#### pandas中的条件函数 : .isin, .str.startswith, .groupby.filter(lec04)

In [30]:
# .isin示例
names = ["Mara", "Helen", "Ruth", "Alice"]
babynames[babynames["Name"].isin(names)]

,State,Sex,Year,Name,Count
1,CA,F,1910,Helen,239
5,CA,F,1910,Ruth,128
7,CA,F,1910,Alice,118
235,CA,F,1911,Helen,209
237,CA,F,1911,Ruth,156
...,...,...,...,...,...
236557,CA,F,2022,Mara,47
245944,CA,M,1925,Alice,7
249066,CA,M,1930,Helen,5
255877,CA,M,1942,Helen,6


In [34]:
# .str.startswith示例
babynames[babynames["Name"].str.startswith("N")]
# babynames.loc[babynames["Name"].str.startswith("N") , "Name"]

,State,Sex,Year,Name,Count
76,CA,F,1910,Norma,23
83,CA,F,1910,Nellie,20
127,CA,F,1910,Nina,11
198,CA,F,1910,Nora,6
310,CA,F,1911,Nellie,23
...,...,...,...,...,...
407319,CA,M,2022,Nilan,5
407320,CA,M,2022,Niles,5
407321,CA,M,2022,Nolen,5
407322,CA,M,2022,Noriel,5


## 修改Dataframe

添加列

In [37]:
# 用[]创建新列，创建时需传入序列对象，或以数组形式传入
# 得到序列对象
babynames_name_lengths = babynames["Name"].str.len()

# 用[]创建新列后传入序列对象
babynames["Name_lengths"] = babynames_name_lengths

In [36]:
babynames.head()

,State,Sex,Year,Name,Count,Name_lengths
0,CA,F,1910,Mary,295,4
1,CA,F,1910,Helen,239,5
2,CA,F,1910,Dorothy,220,7
3,CA,F,1910,Margaret,163,8
4,CA,F,1910,Frances,134,7


重命名列：.rename(columns={"原列名":"新列名"})，参数为字典

In [40]:
babynames = babynames.rename(columns={"Lengths": "Length"})

删除列/行：.drop()(默认删除行，如果要删除整列，要特别说明axis = columns)<br>
**在删除行/列时，实际上是得到了副本，原DataFrame并没有被改变**<br>
 注：babynames = babynames.drop()就能让原DataFrame改变<br>
 **或使用inplace参数=True，即为在原DataFrame上进行修改**

In [49]:
babynames = babynames.drop(columns=["Length"])

# babynames = babynames.drop("Length", axis="columns")